In [1]:
import pandas as pd
import numpy as np

# load surface panel
surface = pd.read_csv("data/processed/surface_panel.csv", index_col="Date")
surface.index = pd.to_datetime(surface.index)

# load filtered options data for OI features
df = pd.read_csv("data/processed/master_with_iv.csv")
df["Date"] = pd.to_datetime(df["Date"])

print("Surface panel:", surface.shape)
print("Options data:", df.shape)
print("\nSurface columns:", surface.columns.tolist())

Surface panel: (101, 42)
Options data: (33893, 14)

Surface columns: ['CE_M1_D1', 'CE_M1_D2', 'CE_M1_D3', 'CE_M1_D4', 'CE_M2_D1', 'CE_M2_D2', 'CE_M2_D3', 'CE_M2_D4', 'CE_M3_D1', 'CE_M3_D2', 'CE_M3_D3', 'CE_M3_D4', 'CE_M4_D1', 'CE_M4_D2', 'CE_M4_D3', 'CE_M4_D4', 'CE_M5_D1', 'CE_M5_D2', 'CE_M5_D3', 'CE_M5_D4', 'PE_M1_D1', 'PE_M1_D2', 'PE_M1_D3', 'PE_M1_D4', 'PE_M2_D1', 'PE_M2_D2', 'PE_M2_D3', 'PE_M2_D4', 'PE_M3_D1', 'PE_M3_D2', 'PE_M3_D3', 'PE_M3_D4', 'PE_M4_D1', 'PE_M4_D2', 'PE_M4_D3', 'PE_M4_D4', 'PE_M5_D1', 'PE_M5_D2', 'PE_M5_D3', 'PE_M5_D4', 'Spot', 'VIX']


In [2]:
atm_ce_cols = ["CE_M3_D1","CE_M3_D2","CE_M3_D3","CE_M3_D4"]
atm_pe_cols = ["PE_M3_D1","PE_M3_D2","PE_M3_D3","PE_M3_D4"]

surface["ATM_IV"]        = surface[atm_ce_cols + atm_pe_cols].mean(axis=1)
surface["ATM_IV_1d_chg"] = surface["ATM_IV"].diff(1)
surface["ATM_IV_5d_chg"] = surface["ATM_IV"].diff(5)

surface["Skew_D1"] = surface["PE_M3_D1"] - surface["CE_M3_D1"]
surface["Skew_D2"] = surface["PE_M3_D2"] - surface["CE_M3_D2"]

surface["TermSlope_CE"] = surface["CE_M3_D1"] - surface["CE_M3_D4"]
surface["TermSlope_PE"] = surface["PE_M3_D1"] - surface["PE_M3_D4"]

surface["SmileWidth_CE"] = surface["CE_M1_D1"] - surface["CE_M3_D1"]
surface["SmileWidth_PE"] = surface["PE_M1_D1"] - surface["PE_M3_D1"]

print("Surface features added:")
print(surface[["ATM_IV","ATM_IV_1d_chg","Skew_D1","TermSlope_CE","SmileWidth_CE"]].head(8).round(4))

Surface features added:
            ATM_IV  ATM_IV_1d_chg  Skew_D1  TermSlope_CE  SmileWidth_CE
Date                                                                   
2024-01-01  0.1405            NaN  -0.0493        0.0059         0.2003
2024-01-02  0.1366        -0.0038   0.0168       -0.0112         0.1869
2024-01-03  0.1289        -0.0078   0.0329       -0.0205         0.1527
2024-01-04  0.1210        -0.0078  -0.0171       -0.0291         0.0739
2024-01-05  0.1128        -0.0083  -0.0021       -0.0285         0.0889
2024-01-08  0.1299         0.0171  -0.0061       -0.0048         0.0541
2024-01-09  0.1284        -0.0015  -0.0011       -0.0043         0.0849
2024-01-10  0.1289         0.0005  -0.0633        0.0247         0.3945


In [3]:
surface["Nifty_Return"]    = surface["Spot"].pct_change(1)
surface["Nifty_5d_Return"] = surface["Spot"].pct_change(5)
surface["RealVol_5d"]      = surface["Nifty_Return"].rolling(5).std() * np.sqrt(252)
surface["RealVol_10d"]     = surface["Nifty_Return"].rolling(10).std() * np.sqrt(252)
surface["VIX_1d_chg"]      = surface["VIX"].diff(1)
surface["VIX_5d_chg"]      = surface["VIX"].diff(5)
surface["IV_Premium"]      = surface["ATM_IV"] - surface["RealVol_5d"]

print("Market features added:")
print(surface[["Nifty_Return","RealVol_5d","VIX_1d_chg","IV_Premium"]].head(8).round(4))

Market features added:
            Nifty_Return  RealVol_5d  VIX_1d_chg  IV_Premium
Date                                                        
2024-01-01           NaN         NaN         NaN         NaN
2024-01-02       -0.0035         NaN        0.00         NaN
2024-01-03       -0.0069         NaN       -0.48         NaN
2024-01-04        0.0066         NaN       -0.77         NaN
2024-01-05        0.0024         NaN       -0.70         NaN
2024-01-08       -0.0091      0.1032        0.83      0.0267
2024-01-09        0.0015      0.1050       -0.20      0.0234
2024-01-10        0.0034      0.0943       -0.29      0.0346


In [4]:
daily_oi = df.groupby(["Date","OptionType"])["OI"].sum().unstack()
daily_oi.columns = ["OI_CE","OI_PE"]

daily_oi["PCR"]        = daily_oi["OI_PE"] / daily_oi["OI_CE"]
daily_oi["OI_CE_chg"]  = daily_oi["OI_CE"].pct_change(1)
daily_oi["OI_PE_chg"]  = daily_oi["OI_PE"].pct_change(1)

surface = surface.merge(daily_oi, on="Date", how="left")

print("OI features added:")
print(surface[["OI_CE","OI_PE","PCR","OI_CE_chg"]].head(8).round(4))

OI features added:
                OI_CE      OI_PE     PCR  OI_CE_chg
Date                                               
2024-01-01  108718550   94040100  0.8650        NaN
2024-01-02  135242100  102472650  0.7577     0.2440
2024-01-03  184921500  130071950  0.7034     0.3673
2024-01-04   62803200   62506850  0.9953    -0.6604
2024-01-05   90139250   82856650  0.9192     0.4353
2024-01-08  121724900   92357700  0.7587     0.3504
2024-01-09  128235950  101982450  0.7953     0.0535
2024-01-10  151460100  141727000  0.9357     0.1811


In [5]:
surface["DayOfWeek"] = surface.index.dayofweek
surface["IsExpiry"]  = (surface.index.dayofweek == 3).astype(int)

surface["DaysSinceExpiry"] = 0
last_expiry = -1
for i, (date, row) in enumerate(surface.iterrows()):
    if row["IsExpiry"] == 1:
        last_expiry = i
    surface.loc[date, "DaysSinceExpiry"] = i - last_expiry if last_expiry >= 0 else 0

surface["Month"] = surface.index.month

print("Calendar features added:")
print(surface[["DayOfWeek","IsExpiry","DaysSinceExpiry","Month"]].head(10))

Calendar features added:
            DayOfWeek  IsExpiry  DaysSinceExpiry  Month
Date                                                   
2024-01-01          0         0                0      1
2024-01-02          1         0                0      1
2024-01-03          2         0                0      1
2024-01-04          3         1                0      1
2024-01-05          4         0                1      1
2024-01-08          0         0                2      1
2024-01-09          1         0                3      1
2024-01-10          2         0                4      1
2024-01-11          3         1                0      1
2024-01-12          4         0                1      1


In [6]:
for col in ["ATM_IV","Skew_D1","TermSlope_CE","PCR","VIX"]:
    surface[f"{col}_lag1"] = surface[col].shift(1)
    surface[f"{col}_lag5"] = surface[col].shift(5)

print("Lag features added.")
print("Total columns now:", len(surface.columns))
print("\nAll feature columns:")
feature_cols = [c for c in surface.columns if c not in
    [f"{opt}_M{m}_D{d}"
     for opt in ["CE","PE"]
     for m in range(1,6)
     for d in range(1,5)]]
print(feature_cols)

Lag features added.
Total columns now: 77

All feature columns:
['Spot', 'VIX', 'ATM_IV', 'ATM_IV_1d_chg', 'ATM_IV_5d_chg', 'Skew_D1', 'Skew_D2', 'TermSlope_CE', 'TermSlope_PE', 'SmileWidth_CE', 'SmileWidth_PE', 'Nifty_Return', 'Nifty_5d_Return', 'RealVol_5d', 'RealVol_10d', 'VIX_1d_chg', 'VIX_5d_chg', 'IV_Premium', 'OI_CE', 'OI_PE', 'PCR', 'OI_CE_chg', 'OI_PE_chg', 'DayOfWeek', 'IsExpiry', 'DaysSinceExpiry', 'Month', 'ATM_IV_lag1', 'ATM_IV_lag5', 'Skew_D1_lag1', 'Skew_D1_lag5', 'TermSlope_CE_lag1', 'TermSlope_CE_lag5', 'PCR_lag1', 'PCR_lag5', 'VIX_lag1', 'VIX_lag5']


In [7]:
surface_features = surface.dropna()

print("Shape before dropna:", surface.shape)
print("Shape after dropna:", surface_features.shape)
print("Rows dropped:", len(surface) - len(surface_features))

surface_features.to_csv("data/processed/features_panel.csv")
print("\nSaved to data/processed/features_panel.csv")
print("Final shape:", surface_features.shape)
print("\nDate range:", surface_features.index[0], "to", surface_features.index[-1])
print("\nSample features:")
print(surface_features[["ATM_IV","Skew_D1","PCR","RealVol_5d","IV_Premium","IsExpiry"]].head(8).round(4))

Shape before dropna: (101, 77)
Shape after dropna: (91, 77)
Rows dropped: 10

Saved to data/processed/features_panel.csv
Final shape: (91, 77)

Date range: 2024-01-15 00:00:00 to 2024-05-31 00:00:00

Sample features:
            ATM_IV  Skew_D1     PCR  RealVol_5d  IV_Premium  IsExpiry
Date                                                                 
2024-01-15  0.1400  -0.0052  1.1316      0.0741      0.0659         0
2024-01-16  0.1372   0.0182  0.9546      0.0931      0.0442         0
2024-01-17  0.1674   0.0101  0.6839      0.2043     -0.0370         0
2024-01-18  0.1467  -0.0363  0.7342      0.2061     -0.0594         1
2024-01-19  0.1418  -0.0254  0.6768      0.1916     -0.0498         0
2024-01-23  0.1727   0.0853  0.6928      0.1834     -0.0107         0
2024-01-24  0.1618  -0.0341  0.7528      0.2243     -0.0626         0
2024-01-25  0.1457  -0.0323  0.8831      0.1777     -0.0320         1
